<a href="https://colab.research.google.com/github/manavkashyap2453-cell/Bank-Fraud-Detection/blob/main/preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Initial setup**

In [2]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import os

import warnings
warnings.filterwarnings('ignore')

In [3]:
path='/content/Bank_Transaction_Fraud_Detection.csv'
flag=os.path.exists(path)

try:
  df=pd.read_csv(path)
except:
  print("Dataset not found")

# **Drop unnecessary columns**

In [4]:
columns_to_drop = [
    'Customer_ID',
    'Customer_Name',
    'Merchant_ID',
    'Customer_Email',
    'Customer_Contact',
    'Transaction_ID',
    'Transaction_Description',
    'Transaction_Location',
    'Bank_Branch',
    'Transaction_Currency'
]

df = df.drop(columns=columns_to_drop)

In [6]:
print(df.shape)
print(df.columns)

(200000, 14)
Index(['Gender', 'Age', 'State', 'City', 'Account_Type', 'Transaction_Date',
       'Transaction_Time', 'Transaction_Amount', 'Transaction_Type',
       'Merchant_Category', 'Account_Balance', 'Transaction_Device',
       'Device_Type', 'Is_Fraud'],
      dtype='object')


# **Feature engineering**

**Date**

In [7]:
df["Transaction_Date"] = pd.to_datetime(df["Transaction_Date"])

df["Day_of_Week"] = df["Transaction_Date"].dt.day_name()

In [8]:
df[["Transaction_Date", "Day_of_Week"]].head()

,Transaction_Date,Day_of_Week
0,2025-01-23,Thursday
1,2025-01-11,Saturday
2,2025-01-25,Saturday
3,2025-01-19,Sunday
4,2025-01-30,Thursday


**Time**

In [9]:
df["Transaction_Time"] = pd.to_datetime(df["Transaction_Time"])

df["Hour"] = df["Transaction_Time"].dt.hour

In [11]:
df[['Transaction_Time','Hour']].head()

,Transaction_Time,Hour
0,2026-07-07 16:04:07,16
1,2026-07-07 17:14:53,17
2,2026-07-07 03:09:52,3
3,2026-07-07 12:27:02,12
4,2026-07-07 18:30:46,18


**Transaction amount and account balance**

In [12]:
df["Amount_Balance_Ratio"] = (
    df["Transaction_Amount"] / df["Account_Balance"]
)

In [13]:
df[[
    "Transaction_Amount",
    "Account_Balance",
    "Amount_Balance_Ratio"
]].head()

,Transaction_Amount,Account_Balance,Amount_Balance_Ratio
0,32415.45,74557.27,0.434772
1,43622.60,74622.66,0.584576
2,63062.56,66817.99,0.943796
3,14000.72,58177.08,0.240657
4,18335.16,16108.56,1.138225


**Drop original date and time columns**

In [14]:
df = df.drop(columns=["Transaction_Date", "Transaction_Time"])

In [15]:
df.columns

Index(['Gender', 'Age', 'State', 'City', 'Account_Type', 'Transaction_Amount',
       'Transaction_Type', 'Merchant_Category', 'Account_Balance',
       'Transaction_Device', 'Device_Type', 'Is_Fraud', 'Day_of_Week', 'Hour',
       'Amount_Balance_Ratio'],
      dtype='object')

# **Train test split**

In [16]:
X = df.drop(columns="Is_Fraud")
y = df["Is_Fraud"]

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [18]:
print(X_train.shape)
print(X_test.shape)

print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(160000, 14)
(40000, 14)
Is_Fraud
0    0.949562
1    0.050438
Name: proportion, dtype: float64
Is_Fraud
0    0.94955
1    0.05045
Name: proportion, dtype: float64


# **Identify numerical and categorica columns**

In [19]:
numerical_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()

categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()

In [20]:
print("Numerical Columns:")
print(numerical_cols)

print("\nCategorical Columns:")
print(categorical_cols)

Numerical Columns:
['Age', 'Transaction_Amount', 'Account_Balance', 'Amount_Balance_Ratio']

Categorical Columns:
['Gender', 'State', 'City', 'Account_Type', 'Transaction_Type', 'Merchant_Category', 'Transaction_Device', 'Device_Type', 'Day_of_Week']


# **Preprocessing Pipeline**

In [21]:
from sklearn.preprocessing import StandardScaler

numerical_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

In [22]:
from sklearn.preprocessing import OneHotEncoder

categorical_pipeline = Pipeline(
    steps=[
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

# **Column the pipelines**

In [23]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numerical_cols),
        ("cat", categorical_pipeline, categorical_cols)
    ]
)

#**Apply the pipeline to data**

In [24]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

# **Handling the imbalance**

**Approach 1: Baseline (No balancing)**

Train the model on the original data.

Why?

It gives a benchmark.
Sometimes models like Random Forest or XGBoost perform well even without resampling.

**Approach 2: Class Weights**

Instead of changing the data, tell the model that fraud cases are more important.

Examples:

*  Logistic Regression → class_weight="balanced"
*  Random Forest → class_weight="balanced"
*  XGBoost → scale_pos_weight

Pros:

No synthetic data.
Often works very well.
Commonly used in production.


**Approach 3: SMOTE**

Generate synthetic fraud samples only on the training data.

Pros:

Gives the model more fraud examples to learn from.
Frequently improves recall.

Cons:

Can increase false positives.
May not always outperform class weighting.

**SMOTE**

In [25]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)

X_train_balanced, y_train_balanced = smote.fit_resample(
    X_train_processed,
    y_train
)

In [26]:
print(y_train.value_counts())

print(y_train_balanced.value_counts())

Is_Fraud
0    151930
1      8070
Name: count, dtype: int64
Is_Fraud
0    151930
1    151930
Name: count, dtype: int64


# **Saving the artifacts**

In [30]:
import os
import joblib


def save_training_artifacts(
    artifact_dir,
    preprocessor,
    X_train_processed,
    X_test_processed,
    X_train_balanced,
    y_train,
    y_test,
    y_train_balanced
):
    """
    Save all artifacts required during model development/training.
    """

    os.makedirs(artifact_dir, exist_ok=True)

    artifacts = {
        "preprocessor.pkl": preprocessor,
        "X_train_processed.pkl": X_train_processed,
        "X_test_processed.pkl": X_test_processed,
        "X_train_balanced.pkl": X_train_balanced,
        "y_train.pkl": y_train,
        "y_test.pkl": y_test,
        "y_train_balanced.pkl": y_train_balanced,
    }

    for filename, artifact in artifacts.items():
        filepath = os.path.join(artifact_dir, filename)
        joblib.dump(artifact, filepath)

    print(f"Successfully saved {len(artifacts)} artifacts to '{artifact_dir}'.")

In [31]:
save_training_artifacts(
    artifact_dir="artifacts",
    preprocessor=preprocessor,
    X_train_processed=X_train_processed,
    X_test_processed=X_test_processed,
    X_train_balanced=X_train_balanced,
    y_train=y_train,
    y_test=y_test,
    y_train_balanced=y_train_balanced
)

Successfully saved 7 artifacts to 'artifacts'.
